# Siglet-Qubit Simulation: Phase 2

## Falsifiability Protocols & Null Hypothesis Testing

**Date:** 2025-11-28  
**Author:** Luiz Frias  
**Version:** 2.0.0

---

## Executive Summary

This notebook implements rigorous falsifiability protocols to validate or refute emergence claims from Phase 1.

**Core Hypothesis (H₁):**
> Emergent cluster structure in siglet truth-decay trajectories is not an artifact of parameterization, noise, or sampling bias, but reflects genuine symbolic dynamical regimes.

**Null Hypothesis (H₀):**
> Observed structure is indistinguishable from clusters produced by randomized, permuted, or noise-only controls.

---

## Experimental Design

| Baseline | Description | Attack Vector |
|----------|-------------|---------------|
| **A** | Random siglets | DTW mathematical defaults |
| **B** | Shuffled trajectories | Statistical shape coincidence |
| **C** | Noise-only | Variance structure sensitivity |
| **D** | ε-collapsed | Ethical operator dependency |
| **E** | τ-constant | Exponential geometry artifact |
| **F** | Synthetic monotonic | Monotone trend artifact |

**Proof Gates:**
1. real_silhouette > max(baseline_silhouettes)
2. real_cluster_persistence > max(baseline_persistence)
3. KS p-value < 0.01 for DTW distributions (all baselines)
4. Noise robustness > max(baseline_robustness)

---

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Load Phase 1/1.2 Data](#2-load-phase-112-data)
3. [Generate Null Baselines](#3-generate-null-baselines)
4. [Compute DTW Matrices](#4-compute-dtw-matrices)
5. [Perform Clustering](#5-perform-clustering)
6. [Compute Metrics](#6-compute-metrics)
7. [Statistical Comparison](#7-statistical-comparison)
8. [Outcome Determination](#8-outcome-determination)
9. [Export Results](#9-export-results)

---

## 1. Environment Setup

In [ ]:
# Standard library
import os
import time
import json
import warnings
from pathlib import Path

# Data science
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist, squareform

# Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

# Machine Learning
from sklearn.cluster import SpectralClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
)

# Time series
from fastdtw import fastdtw

# Experiment tracking
import mlflow

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment loaded successfully.")

In [ ]:
# Project paths
ROOT_DIR = Path("/Users/luizfrias/CursorAI/data-science/siglet_architecture")
DATA_DIR = ROOT_DIR / "data" / "processed"
INTERIM_DIR = ROOT_DIR / "data" / "interim"
PHASE2_DTW_DIR = INTERIM_DIR / "phase2_dtw"
FIGURES_DIR = ROOT_DIR / "reports" / "figures"
NOTES_DIR = ROOT_DIR / "reports" / "notes"

# Create directories
for d in [DATA_DIR, INTERIM_DIR, PHASE2_DTW_DIR, FIGURES_DIR, NOTES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# MLflow setup
mlflow.set_experiment("siglet_phase2_falsifiability")

print(f"Root: {ROOT_DIR}")
print(f"Phase 2 DTW cache: {PHASE2_DTW_DIR}")

In [ ]:
# Global parameters
N_CLUSTERS = 5
T_MAX = 10
BOOTSTRAP_ITERATIONS = 100
SIGNIFICANCE_LEVEL = 0.01

print(f"Parameters:")
print(f"  N_CLUSTERS: {N_CLUSTERS}")
print(f"  T_MAX: {T_MAX}")
print(f"  BOOTSTRAP_ITERATIONS: {BOOTSTRAP_ITERATIONS}")
print(f"  SIGNIFICANCE_LEVEL: {SIGNIFICANCE_LEVEL}")

---

## 2. Load Phase 1/1.2 Data

Preferentially loads Phase 1.2 data (with full ε-vector structure) if available, otherwise falls back to Phase 1 data.

In [ ]:
# Load real siglet data - prefer Phase 1.2 if available
print("Loading siglet data...")

# Check for Phase 1.2 data first (improved full ε-vector structure)
phase1_2_curves = DATA_DIR / "decay_curves_phase1.2.npy"
phase1_2_params = DATA_DIR / "curve_params_phase1.2.npy"

if phase1_2_curves.exists():
    print(
        "  Found Phase 1.2 data - loading improved curves with full ε-vector structure"
    )
    decay_curves_real = np.load(phase1_2_curves)
    curve_params = np.load(phase1_2_params)
    DATA_SOURCE = "Phase 1.2"
else:
    print("  Phase 1.2 data not found - falling back to Phase 1 data")
    decay_curves_real = np.load(DATA_DIR / "decay_curves.npy")
    curve_params = np.load(DATA_DIR / "curve_params.npy")
    DATA_SOURCE = "Phase 1"

# Load spectral labels if available, otherwise we'll compute them
spectral_labels_file = DATA_DIR / "spectral_labels.npy"
if spectral_labels_file.exists():
    spectral_labels_real = np.load(spectral_labels_file)
else:
    spectral_labels_real = None  # Will be computed during clustering

N_SIGLETS = len(decay_curves_real)

print(f"\nData Source: {DATA_SOURCE}")
print(f"Loaded {N_SIGLETS} real siglets")
print(f"Decay curve shape: {decay_curves_real.shape}")
if spectral_labels_real is not None:
    print(f"Cluster distribution: {np.bincount(spectral_labels_real)}")
else:
    print("Spectral labels will be computed during clustering")

---

## 3. Generate Null Baselines

We generate 6 null baselines, each attacking a different potential artifact source.

In [ ]:
def generate_baseline_A_random(n_siglets, t_max):
    """
    Baseline A: Random Siglets

    Sample random (r, τ, ε, μ, c) destroying all structured coupling.
    Purpose: Ensures clusters aren't mathematical defaults of DTW.
    """
    curves = []
    for _ in range(n_siglets):
        # Random parameters
        r = np.random.uniform(-1, 1)  # Resonance
        tau = np.random.uniform(0.5, 1.0)  # Coherence
        eps = np.random.uniform(0.5, 1.5)  # ε-norm
        mu = np.random.uniform(0.5, 1.0)  # Modality
        c = np.random.uniform(0.5, 1.0)  # Compression

        # Generate trajectory
        curve = [r * (tau**t) * eps * mu * c for t in range(t_max + 1)]
        curves.append(curve)

    return np.array(curves)


def generate_baseline_B_shuffled(real_curves):
    """
    Baseline B: Shuffled Trajectories

    Apply per-trajectory permutation, preserving value distribution.
    Purpose: Ensures clusters aren't dominated by statistical shape coincidence.
    """
    shuffled = []
    for curve in real_curves:
        shuffled_curve = np.random.permutation(curve)
        shuffled.append(shuffled_curve)
    return np.array(shuffled)


def generate_baseline_C_noise(n_siglets, t_max, variance=None):
    """
    Baseline C: Noise-Only Trajectories

    T(t) = noise, matching variance of real data.
    Purpose: Checks sensitivity to variance structure.
    """
    if variance is None:
        variance = 0.1

    curves = np.random.normal(0, np.sqrt(variance), (n_siglets, t_max + 1))
    return curves


def generate_baseline_D_epsilon_collapsed(n_siglets, t_max):
    """
    Baseline D: ε-Collapsed

    Set ε = constant, run simulation.
    Purpose: Verifies clusters depend on ethical operator space.
    """
    curves = []
    for _ in range(n_siglets):
        theta = np.random.uniform(0, np.pi)
        tau = np.random.uniform(0.8, 1.0)
        eps = 1.0  # CONSTANT
        mu = 0.8
        c = 0.7

        r = np.cos(theta)
        curve = [r * (tau**t) * eps * mu * c for t in range(t_max + 1)]
        curves.append(curve)

    return np.array(curves)


def generate_baseline_E_tau_constant(n_siglets, t_max):
    """
    Baseline E: τ-Constant

    Set all τ to constant, removing temporal structure.
    Purpose: Ensures stability isn't just exponential decay geometry.
    """
    curves = []
    for _ in range(n_siglets):
        theta = np.random.uniform(0, np.pi)
        tau = 0.95  # CONSTANT
        eps = np.random.uniform(0.8, 1.2)
        mu = 0.8
        c = 0.7

        r = np.cos(theta)
        curve = [r * (tau**t) * eps * mu * c for t in range(t_max + 1)]
        curves.append(curve)

    return np.array(curves)


def generate_baseline_F_polynomial(n_siglets, t_max):
    """
    Baseline F: Synthetic Monotonic (Polynomial Decay)

    Use polynomial decays with small noise.
    Purpose: Ensures clustering isn't just a function of monotone trend shape.
    """
    curves = []
    for _ in range(n_siglets):
        # Random polynomial coefficients
        a = np.random.uniform(0.3, 0.7)
        b = np.random.uniform(-0.1, 0.1)
        sign = np.random.choice([-1, 1])

        t_vals = np.arange(t_max + 1)
        curve = sign * (a - b * t_vals / t_max) + np.random.normal(0, 0.01, t_max + 1)
        curves.append(curve)

    return np.array(curves)

In [ ]:
# Generate all baselines
with mlflow.start_run(run_name="generate_null_baselines"):
    print("Generating null baselines...")
    start_time = time.time()

    # Compute variance of real data for baseline C
    real_variance = np.var(decay_curves_real)

    baselines = {
        "A_random": generate_baseline_A_random(N_SIGLETS, T_MAX),
        "B_shuffled": generate_baseline_B_shuffled(decay_curves_real),
        "C_noise": generate_baseline_C_noise(N_SIGLETS, T_MAX, real_variance),
        "D_eps_collapsed": generate_baseline_D_epsilon_collapsed(N_SIGLETS, T_MAX),
        "E_tau_constant": generate_baseline_E_tau_constant(N_SIGLETS, T_MAX),
        "F_polynomial": generate_baseline_F_polynomial(N_SIGLETS, T_MAX),
    }

    # Add real data to the dictionary
    all_datasets = {"real": decay_curves_real, **baselines}

    computation_time = time.time() - start_time
    mlflow.log_metric("baseline_generation_time", computation_time)

    print(f"\nBaseline generation completed in {computation_time:.2f}s")
    print("\nBaseline statistics:")
    print("-" * 60)
    for name, data in all_datasets.items():
        print(
            f"  {name:20s}: shape={data.shape}, range=[{data.min():.3f}, {data.max():.3f}]"
        )

In [ ]:
# Visualize sample trajectories from each baseline
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, (name, data) in enumerate(all_datasets.items()):
    ax = axes[idx]

    # Plot 10 random samples
    sample_idx = np.random.choice(len(data), min(10, len(data)), replace=False)
    for i in sample_idx:
        ax.plot(range(T_MAX + 1), data[i], alpha=0.7)

    ax.set_title(f"{name}")
    ax.set_xlabel("Time (t)")
    ax.set_ylabel("Truth Score")
    ax.grid(True, alpha=0.3)

# Hide empty subplot
axes[-1].axis("off")

plt.suptitle("Sample Trajectories from Each Dataset", fontsize=14)
plt.tight_layout()

fig_path = FIGURES_DIR / "phase2_baseline_samples.png"
plt.savefig(fig_path, dpi=300)
plt.show()

---

## 4. Compute DTW Matrices

In [ ]:
def compute_dtw_matrix_sampled(curves, sample_size=300, seed=42):
    """
    Compute DTW distance matrix on a sample for efficiency.

    Returns:
        dtw_matrix: Distance matrix
        sample_indices: Indices of sampled curves
    """
    np.random.seed(seed)

    n = len(curves)
    if n <= sample_size:
        sample_idx = np.arange(n)
    else:
        sample_idx = np.random.choice(n, sample_size, replace=False)

    sampled_curves = curves[sample_idx]
    n_sample = len(sampled_curves)

    dtw_matrix = np.zeros((n_sample, n_sample))

    for i in range(n_sample):
        for j in range(i + 1, n_sample):
            d, _ = fastdtw(sampled_curves[i], sampled_curves[j])
            dtw_matrix[i, j] = d
            dtw_matrix[j, i] = d

    return dtw_matrix, sample_idx

In [ ]:
# Compute DTW matrices for all datasets
with mlflow.start_run(run_name="compute_dtw_matrices"):
    print("Computing DTW matrices for all datasets...")
    start_time = time.time()

    SAMPLE_SIZE = 300  # For efficiency
    mlflow.log_param("dtw_sample_size", SAMPLE_SIZE)

    dtw_matrices = {}
    sample_indices = {}

    for name, data in all_datasets.items():
        print(f"  Processing {name}...")
        dtw_mat, sample_idx = compute_dtw_matrix_sampled(data, SAMPLE_SIZE)
        dtw_matrices[name] = dtw_mat
        sample_indices[name] = sample_idx

        # Save to cache
        np.save(PHASE2_DTW_DIR / f"dtw_{name}.npy", dtw_mat)
        np.save(PHASE2_DTW_DIR / f"sample_idx_{name}.npy", sample_idx)

        print(f"    DTW matrix shape: {dtw_mat.shape}")
        print(f"    Distance range: [{dtw_mat.min():.3f}, {dtw_mat.max():.3f}]")

    computation_time = time.time() - start_time
    mlflow.log_metric("dtw_computation_time", computation_time)

    print(f"\nDTW computation completed in {computation_time:.2f}s")

---

## 5. Perform Clustering

In [ ]:
def perform_spectral_clustering(dtw_matrix, n_clusters, seed=42):
    """
    Perform spectral clustering on DTW distance matrix.
    """
    # Normalize
    dtw_norm = dtw_matrix / (np.max(dtw_matrix) + 1e-10)

    # Convert to affinity
    affinity = np.exp(-dtw_norm)

    # Cluster
    clustering = SpectralClustering(
        n_clusters=n_clusters, affinity="precomputed", random_state=seed
    )
    labels = clustering.fit_predict(affinity)

    return labels

In [ ]:
# Perform clustering on all datasets
with mlflow.start_run(run_name="perform_clustering"):
    print("Performing spectral clustering on all datasets...")

    cluster_labels = {}

    for name, dtw_mat in dtw_matrices.items():
        labels = perform_spectral_clustering(dtw_mat, N_CLUSTERS)
        cluster_labels[name] = labels

        print(f"  {name}: {np.bincount(labels)}")
        mlflow.log_metric(f"n_clusters_{name}", len(np.unique(labels)))

    print("\nClustering completed.")

---

## 6. Compute Metrics

In [ ]:
def compute_clustering_metrics(dtw_matrix, labels):
    """
    Compute comprehensive clustering quality metrics.

    Returns:
        metrics: Dict with silhouette, davies_bouldin, etc.
    """
    metrics = {}

    # Check if we have enough clusters
    n_unique = len(np.unique(labels))
    if n_unique < 2:
        return {
            "silhouette": -1,
            "davies_bouldin": float("inf"),
            "intra_cluster_dist": 0,
            "inter_cluster_dist": 0,
            "density_ratio": 0,
        }

    # Silhouette score (higher is better)
    try:
        metrics["silhouette"] = silhouette_score(
            dtw_matrix, labels, metric="precomputed"
        )
    except:
        metrics["silhouette"] = -1

    # Davies-Bouldin index (lower is better)
    # Need to use features, so we'll use MDS embedding
    try:
        from sklearn.manifold import MDS

        mds = MDS(
            n_components=2,
            dissimilarity="precomputed",
            random_state=RANDOM_SEED,
            max_iter=100,
        )
        embedding = mds.fit_transform(dtw_matrix)
        metrics["davies_bouldin"] = davies_bouldin_score(embedding, labels)
    except:
        metrics["davies_bouldin"] = float("inf")

    # Intra-cluster distance (mean distance within clusters)
    intra_dists = []
    for cluster_id in np.unique(labels):
        mask = labels == cluster_id
        if np.sum(mask) > 1:
            cluster_dtw = dtw_matrix[np.ix_(mask, mask)]
            intra_dists.append(
                np.mean(cluster_dtw[np.triu_indices(len(cluster_dtw), k=1)])
            )
    metrics["intra_cluster_dist"] = np.mean(intra_dists) if intra_dists else 0

    # Inter-cluster distance (mean distance between clusters)
    inter_dists = []
    unique_labels = np.unique(labels)
    for i, c1 in enumerate(unique_labels):
        for c2 in unique_labels[i + 1 :]:
            mask1 = labels == c1
            mask2 = labels == c2
            inter_dtw = dtw_matrix[np.ix_(mask1, mask2)]
            inter_dists.append(np.mean(inter_dtw))
    metrics["inter_cluster_dist"] = np.mean(inter_dists) if inter_dists else 0

    # Density ratio (inter/intra, higher is better)
    if metrics["intra_cluster_dist"] > 0:
        metrics["density_ratio"] = (
            metrics["inter_cluster_dist"] / metrics["intra_cluster_dist"]
        )
    else:
        metrics["density_ratio"] = 0

    return metrics

In [ ]:
def compute_bootstrap_stability(curves, dtw_matrix, base_labels, n_iterations=100):
    """
    Compute cluster stability via subsample resampling.

    Uses subsampling WITHOUT replacement to avoid indexing ambiguity
    that arises with bootstrap (with replacement) when mapping labels
    back to original indices.

    Returns:
        stability_score: Mean ARI between base and subsample clusterings
    """
    ari_scores = []
    n = len(curves)
    subsample_size = int(0.8 * n)  # Use 80% subsample

    for iteration in range(n_iterations):
        # Subsample WITHOUT replacement - avoids indexing ambiguity
        subsample_idx = np.random.choice(n, subsample_size, replace=False)

        # Extract DTW submatrix for the subsample
        subsample_dtw = dtw_matrix[np.ix_(subsample_idx, subsample_idx)]

        # Cluster the subsample
        subsample_labels = perform_spectral_clustering(subsample_dtw, N_CLUSTERS)

        # Get corresponding base labels for the same indices
        base_subset_labels = base_labels[subsample_idx]

        # Compute ARI between original clustering and subsample clustering
        # for the same set of samples
        if (
            len(np.unique(base_subset_labels)) > 1
            and len(np.unique(subsample_labels)) > 1
        ):
            ari = adjusted_rand_score(base_subset_labels, subsample_labels)
            ari_scores.append(ari)

    return np.mean(ari_scores) if ari_scores else 0

In [ ]:
# Compute all metrics for all datasets
with mlflow.start_run(run_name="compute_all_metrics"):
    print("Computing metrics for all datasets...")
    start_time = time.time()

    all_metrics = {}

    for name in all_datasets.keys():
        print(f"  Processing {name}...")

        dtw_mat = dtw_matrices[name]
        labels = cluster_labels[name]
        curves = all_datasets[name][sample_indices[name]]

        # Basic metrics
        metrics = compute_clustering_metrics(dtw_mat, labels)

        # Bootstrap stability (reduced iterations for speed)
        boot_iters = 30 if name != "real" else BOOTSTRAP_ITERATIONS
        metrics["bootstrap_stability"] = compute_bootstrap_stability(
            curves, dtw_mat, labels, n_iterations=boot_iters
        )

        all_metrics[name] = metrics

        # Log to MLflow
        for metric_name, value in metrics.items():
            if not np.isinf(value):
                mlflow.log_metric(f"{name}_{metric_name}", value)

        print(f"    Silhouette: {metrics['silhouette']:.4f}")
        print(f"    Davies-Bouldin: {metrics['davies_bouldin']:.4f}")
        print(f"    Bootstrap stability: {metrics['bootstrap_stability']:.4f}")

    computation_time = time.time() - start_time
    mlflow.log_metric("metrics_computation_time", computation_time)

    print(f"\nMetrics computation completed in {computation_time:.2f}s")

In [ ]:
# Create metrics comparison table
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df.round(4)

print("\nMetrics Comparison:")
print("=" * 80)
print(metrics_df.to_string())
print("=" * 80)

---

## 7. Statistical Comparison

In [ ]:
def compute_dtw_distribution_ks_test(dtw_real, dtw_baseline):
    """
    Kolmogorov-Smirnov test comparing DTW distance distributions.
    """
    # Extract upper triangular (unique pairs)
    real_dists = dtw_real[np.triu_indices(len(dtw_real), k=1)]
    baseline_dists = dtw_baseline[np.triu_indices(len(dtw_baseline), k=1)]

    # KS test
    statistic, p_value = stats.ks_2samp(real_dists, baseline_dists)

    return statistic, p_value

In [ ]:
# Statistical comparison: Real vs each baseline
with mlflow.start_run(run_name="statistical_comparison"):
    print("Running statistical comparisons...")

    comparison_results = {}
    baseline_names = [name for name in all_datasets.keys() if name != "real"]

    for baseline_name in baseline_names:
        print(f"\n  Comparing real vs {baseline_name}:")

        # KS test on DTW distributions
        ks_stat, ks_p = compute_dtw_distribution_ks_test(
            dtw_matrices["real"], dtw_matrices[baseline_name]
        )

        # Metric differences
        sil_diff = (
            all_metrics["real"]["silhouette"] - all_metrics[baseline_name]["silhouette"]
        )
        db_diff = (
            all_metrics[baseline_name]["davies_bouldin"]
            - all_metrics["real"]["davies_bouldin"]
        )
        stab_diff = (
            all_metrics["real"]["bootstrap_stability"]
            - all_metrics[baseline_name]["bootstrap_stability"]
        )

        comparison_results[baseline_name] = {
            "ks_statistic": ks_stat,
            "ks_p_value": ks_p,
            "ks_significant": ks_p < SIGNIFICANCE_LEVEL,
            "silhouette_diff": sil_diff,
            "real_silhouette_higher": sil_diff > 0,
            "db_diff": db_diff,
            "real_db_lower": db_diff > 0,
            "stability_diff": stab_diff,
            "real_stability_higher": stab_diff > 0,
        }

        print(f"    KS statistic: {ks_stat:.4f}, p-value: {ks_p:.6f}")
        print(
            f"    Silhouette diff: {sil_diff:+.4f} (real {'>' if sil_diff > 0 else '<'} baseline)"
        )
        print(
            f"    Stability diff: {stab_diff:+.4f} (real {'>' if stab_diff > 0 else '<'} baseline)"
        )

        # Log to MLflow
        mlflow.log_metrics(
            {
                f"ks_stat_{baseline_name}": ks_stat,
                f"ks_p_{baseline_name}": ks_p,
                f"sil_diff_{baseline_name}": sil_diff,
                f"stab_diff_{baseline_name}": stab_diff,
            }
        )

In [ ]:
# Visualize DTW distributions comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Get real DTW distribution
real_dists = dtw_matrices["real"][np.triu_indices(len(dtw_matrices["real"]), k=1)]

for idx, baseline_name in enumerate(baseline_names):
    ax = axes[idx]

    baseline_dists = dtw_matrices[baseline_name][
        np.triu_indices(len(dtw_matrices[baseline_name]), k=1)
    ]

    ax.hist(real_dists, bins=30, alpha=0.5, label="Real", density=True)
    ax.hist(baseline_dists, bins=30, alpha=0.5, label=baseline_name, density=True)

    p_val = comparison_results[baseline_name]["ks_p_value"]
    ax.set_title(f"Real vs {baseline_name}\n(KS p={p_val:.4f})")
    ax.set_xlabel("DTW Distance")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("DTW Distance Distributions: Real vs Baselines", fontsize=14)
plt.tight_layout()

fig_path = FIGURES_DIR / "phase2_dtw_baseline_comparison.png"
plt.savefig(fig_path, dpi=300)
plt.show()

---

## 8. Outcome Determination

In [ ]:
# Evaluate proof gates
with mlflow.start_run(run_name="outcome_determination"):
    print("\n" + "=" * 70)
    print("PHASE 2 OUTCOME DETERMINATION")
    print("=" * 70)

    # Extract baseline metrics
    baseline_silhouettes = [all_metrics[name]["silhouette"] for name in baseline_names]
    baseline_stabilities = [
        all_metrics[name]["bootstrap_stability"] for name in baseline_names
    ]
    baseline_dbs = [all_metrics[name]["davies_bouldin"] for name in baseline_names]

    # Proof Gate 1: real_silhouette > max(baseline_silhouettes)
    gate1_passed = all_metrics["real"]["silhouette"] > max(baseline_silhouettes)
    print(f"\nGate 1: Silhouette Score")
    print(f"  Real: {all_metrics['real']['silhouette']:.4f}")
    print(f"  Max Baseline: {max(baseline_silhouettes):.4f}")
    print(f"  Status: {'PASSED ✓' if gate1_passed else 'FAILED ✗'}")

    # Proof Gate 2: real_cluster_persistence > max(baseline_persistence)
    gate2_passed = all_metrics["real"]["bootstrap_stability"] > max(
        baseline_stabilities
    )
    print(f"\nGate 2: Bootstrap Stability")
    print(f"  Real: {all_metrics['real']['bootstrap_stability']:.4f}")
    print(f"  Max Baseline: {max(baseline_stabilities):.4f}")
    print(f"  Status: {'PASSED ✓' if gate2_passed else 'FAILED ✗'}")

    # Proof Gate 3: KS p-value < 0.01 for all baselines
    all_ks_significant = all(
        comparison_results[name]["ks_significant"] for name in baseline_names
    )
    gate3_passed = all_ks_significant
    print(f"\nGate 3: KS Test Significance (p < {SIGNIFICANCE_LEVEL})")
    for name in baseline_names:
        p_val = comparison_results[name]["ks_p_value"]
        sig = comparison_results[name]["ks_significant"]
        print(f"  {name}: p={p_val:.6f} {'✓' if sig else '✗'}")
    print(f"  Status: {'PASSED ✓' if gate3_passed else 'FAILED ✗'}")

    # Proof Gate 4: Davies-Bouldin lower than baselines (or close)
    real_db = all_metrics["real"]["davies_bouldin"]
    valid_baseline_dbs = [db for db in baseline_dbs if not np.isinf(db)]
    if valid_baseline_dbs:
        gate4_passed = real_db < np.median(valid_baseline_dbs)
    else:
        gate4_passed = True
    print(f"\nGate 4: Davies-Bouldin Index")
    print(f"  Real: {real_db:.4f}")
    print(
        f"  Median Baseline: {np.median(valid_baseline_dbs):.4f}"
        if valid_baseline_dbs
        else "  No valid baselines"
    )
    print(f"  Status: {'PASSED ✓' if gate4_passed else 'FAILED ✗'}")

    # Overall verdict
    all_gates_passed = gate1_passed and gate2_passed and gate3_passed and gate4_passed

    print("\n" + "=" * 70)
    if all_gates_passed:
        verdict = "H₀ REJECTED: Hypothesis survives Phase 2"
        verdict_code = "PASSED"
    else:
        verdict = "H₀ NOT REJECTED: Hypothesis enters revision"
        verdict_code = "FAILED"

    print(f"VERDICT: {verdict}")
    print("=" * 70)

    # Log to MLflow
    mlflow.log_metrics(
        {
            "gate1_silhouette_passed": int(gate1_passed),
            "gate2_stability_passed": int(gate2_passed),
            "gate3_ks_passed": int(gate3_passed),
            "gate4_db_passed": int(gate4_passed),
            "all_gates_passed": int(all_gates_passed),
        }
    )

In [ ]:
# Visualize metric comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

dataset_names = list(all_metrics.keys())
colors = ["green" if name == "real" else "steelblue" for name in dataset_names]

# Plot 1: Silhouette Score
ax1 = axes[0]
sil_values = [all_metrics[name]["silhouette"] for name in dataset_names]
bars1 = ax1.bar(dataset_names, sil_values, color=colors)
ax1.axhline(
    y=max(baseline_silhouettes), color="r", linestyle="--", label="Max Baseline"
)
ax1.set_ylabel("Silhouette Score")
ax1.set_title(f"Silhouette Score (Gate 1: {'PASS' if gate1_passed else 'FAIL'})")
ax1.tick_params(axis="x", rotation=45)
ax1.legend()

# Plot 2: Bootstrap Stability
ax2 = axes[1]
stab_values = [all_metrics[name]["bootstrap_stability"] for name in dataset_names]
bars2 = ax2.bar(dataset_names, stab_values, color=colors)
ax2.axhline(
    y=max(baseline_stabilities), color="r", linestyle="--", label="Max Baseline"
)
ax2.set_ylabel("Bootstrap Stability (ARI)")
ax2.set_title(f"Bootstrap Stability (Gate 2: {'PASS' if gate2_passed else 'FAIL'})")
ax2.tick_params(axis="x", rotation=45)
ax2.legend()

# Plot 3: Davies-Bouldin Index
ax3 = axes[2]
db_values = [
    min(all_metrics[name]["davies_bouldin"], 5) for name in dataset_names
]  # Cap at 5 for viz
bars3 = ax3.bar(dataset_names, db_values, color=colors)
if valid_baseline_dbs:
    ax3.axhline(
        y=np.median(valid_baseline_dbs),
        color="r",
        linestyle="--",
        label="Median Baseline",
    )
ax3.set_ylabel("Davies-Bouldin Index (lower is better)")
ax3.set_title(f"Davies-Bouldin Index (Gate 4: {'PASS' if gate4_passed else 'FAIL'})")
ax3.tick_params(axis="x", rotation=45)
ax3.legend()

plt.suptitle(f"Phase 2 Metrics Comparison - Overall: {verdict_code}", fontsize=14)
plt.tight_layout()

fig_path = FIGURES_DIR / "phase2_metric_comparison.png"
plt.savefig(fig_path, dpi=300)
plt.show()

---

## 9. Export Results

In [ ]:
# Export all results
with mlflow.start_run(run_name="export_phase2_results"):
    print("Exporting Phase 2 results...")

    # 1. JSON results
    results_json = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "parameters": {
            "n_siglets": N_SIGLETS,
            "n_clusters": N_CLUSTERS,
            "significance_level": SIGNIFICANCE_LEVEL,
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
        },
        "metrics": {
            k: {kk: float(vv) if not np.isinf(vv) else "inf" for kk, vv in v.items()}
            for k, v in all_metrics.items()
        },
        "comparison_results": {
            k: {
                kk: float(vv) if isinstance(vv, (int, float, np.floating)) else vv
                for kk, vv in v.items()
            }
            for k, v in comparison_results.items()
        },
        "proof_gates": {
            "gate1_silhouette": gate1_passed,
            "gate2_stability": gate2_passed,
            "gate3_ks_test": gate3_passed,
            "gate4_davies_bouldin": gate4_passed,
        },
        "verdict": verdict_code,
        "hypothesis_survives": all_gates_passed,
    }

    with open(DATA_DIR / "phase2_results.json", "w") as f:
        json.dump(results_json, f, indent=2)
    print("  Saved: phase2_results.json")

    # 2. Stability matrix (cluster labels for all datasets)
    stability_matrix = np.array([cluster_labels[name] for name in all_datasets.keys()])
    np.save(DATA_DIR / "phase2_stability_matrix.npy", stability_matrix)
    print("  Saved: phase2_stability_matrix.npy")

    # 3. Metrics CSV
    metrics_df.to_csv(DATA_DIR / "phase2_metrics.csv")
    print("  Saved: phase2_metrics.csv")

    # 4. Summary markdown
    summary_md = (
        f"""
# Phase 2 Falsifiability Results

**Date:** {time.strftime("%Y-%m-%d %H:%M:%S")}

## Verdict: {verdict_code}

{verdict}

## Proof Gate Results

| Gate | Description | Status |
|------|-------------|--------|
| 1 | real_silhouette > max(baseline) | {"PASSED" if gate1_passed else "FAILED"} |
| 2 | real_stability > max(baseline) | {"PASSED" if gate2_passed else "FAILED"} |
| 3 | KS p-value < 0.01 (all baselines) | {"PASSED" if gate3_passed else "FAILED"} |
| 4 | real_DB < median(baseline) | {"PASSED" if gate4_passed else "FAILED"} |

## Metrics Summary

### Real Data
- Silhouette: {all_metrics["real"]["silhouette"]:.4f}
- Davies-Bouldin: {all_metrics["real"]["davies_bouldin"]:.4f}
- Bootstrap Stability: {all_metrics["real"]["bootstrap_stability"]:.4f}

### Baseline Comparison

| Baseline | Silhouette | Stability | KS p-value |
|----------|------------|-----------|------------|
"""
        + "\n".join(
            [
                f"| {name} | {all_metrics[name]['silhouette']:.4f} | {all_metrics[name]['bootstrap_stability']:.4f} | {comparison_results[name]['ks_p_value']:.6f} |"
                for name in baseline_names
            ]
        )
        + f"""

## Interpretation

{"The emergent cluster structure in siglet truth-decay trajectories demonstrates genuine symbolic dynamical regimes that are NOT artifacts of parameterization, noise, or sampling bias." if all_gates_passed else "The hypothesis requires revision. Some aspects of the emergent structure may be artifacts of the simulation methodology."}

## Artifacts Generated

- `phase2_results.json` - Complete results in JSON format
- `phase2_stability_matrix.npy` - Cluster labels for all datasets
- `phase2_metrics.csv` - Metrics comparison table
- `phase2_dtw_baseline_comparison.png` - DTW distribution visualization
- `phase2_metric_comparison.png` - Metrics bar charts

## Next Steps

{"Proceed to Phase 3: Semantic Retro-Labeling and Ethical Axis Interpretation." if all_gates_passed else "Revise the siglet-qubit framework addressing identified weaknesses before proceeding."}
"""
    )

    with open(NOTES_DIR / "phase2_null_vs_real_cluster_metrics.md", "w") as f:
        f.write(summary_md)
    print("  Saved: phase2_null_vs_real_cluster_metrics.md")

    # Log artifacts to MLflow
    mlflow.log_artifact(str(DATA_DIR / "phase2_results.json"))
    mlflow.log_artifact(str(DATA_DIR / "phase2_stability_matrix.npy"))
    mlflow.log_artifact(str(DATA_DIR / "phase2_metrics.csv"))
    mlflow.log_artifact(str(NOTES_DIR / "phase2_null_vs_real_cluster_metrics.md"))
    mlflow.log_artifact(str(FIGURES_DIR / "phase2_dtw_baseline_comparison.png"))
    mlflow.log_artifact(str(FIGURES_DIR / "phase2_metric_comparison.png"))

    print("\nAll Phase 2 results exported successfully.")

---

## End of Phase 2

This notebook has implemented rigorous falsifiability protocols for the Siglet-Qubit framework:

1. **Generated 6 null baselines** attacking different potential artifact sources
2. **Computed DTW matrices** for all datasets
3. **Performed spectral clustering** with consistent parameters
4. **Evaluated comprehensive metrics** (silhouette, Davies-Bouldin, bootstrap stability)
5. **Ran statistical comparisons** (KS tests on DTW distributions)
6. **Determined outcome** via 4 proof gates

The goal was not to "prove right," but to invite being proven wrong with rigor.

---

> *"The siglet-qubit model is a philosophical and architectural commitment to build symbolic systems that evolve ethically—not by instruction, but by coherence."*